# Equifinality of OGGM's dynamical runs

**Experimental.** This notebook is part of a research project: the tools and
the choices in it are not settled.

## The question

To simulate a glacier, OGGM needs parameters which we cannot measure. In
OGGM's usual workflow, they are set in three steps:

1. **MB calibration**: the mass balance (MB) parameters (`melt_f`,
   `prcp_fac`, `temp_bias`) are chosen so that the modelled MB matches the
   geodetic MB 2000-2020 (Hugonnet et al., 2021);
2. **ice thickness inversion**: the ice thickness at the RGI date is computed
   from the MB and the creep parameter of ice (Glen's A). A is chosen so that
   the volume of all the glaciers of a region matches a reference (IceBoost v2
   by default);
3. **dynamic spinup**: a simulation which starts decades before the RGI date
   needs the glacier at that time, which we do not know. OGGM makes one: a
   short run with a constant climate and a temperature bias, chosen so that the
   glacier has the RGI area at the RGI date. OGGM's dynamic calibration
   (`run_dynamic_melt_f_calibration`) also adjusts `melt_f`, so that the
   dynamical run still matches the geodetic MB.

Each step gives *one* value of each parameter. But other combinations would
match the observations just as well: the model is **equifinal**. How many
parameter sets are consistent with what we observe, and how different are the
glaciers (and the runoff) they produce?

To find out, we let five parameters vary at the same time: the three MB
parameters, Glen's A, and a temperature offset which makes the glacier at the
start of the simulation (the "initial state"). We keep the parameter sets
which reproduce three observations: the geodetic MB, the RGI area and the
IceBoost mean ice thickness.

## How: from "keep the plausible runs" to AdaPBS

A simple way to do this, as in GLUE: draw many parameter sets at random (for
example with a Latin hypercube), run the model for each of them, and keep the
"plausible" runs, those close enough to the observations. This is easy to
understand, but most runs are wasted far from the observations, and the
threshold ("close enough") is arbitrary.

AdaPBS (see the first two notebooks) does the same job more efficiently:

- each run gets a **weight**, which says how well it matches the observations,
  given their errors: a "soft" threshold;
- after a first batch of random parameter sets, AdaPBS draws the next batches
  close to the runs with large weights, and stops when enough different runs
  have a large weight.

With a wide prior (here nearly uniform between reasonable bounds), the
weighted runs are the plausible runs. They are what we analyse, here and in
the next notebook (`dynamic_runoff_sensitivity.ipynb`: their runoff, and which
parameters matter for it).

We are *not* trying to find the "best" value of each parameter. A parameter
which the observations cannot constrain is not a failure of the method: it is
one of the results.

AdaPBS is the work of the MuSA authors. If you use it, please cite Aalstad et
al. (2026, GMD, https://doi.org/10.5194/gmd-19-8565-2026) and Alonso-González
et al. (2022, GMD, https://doi.org/10.5194/gmd-15-9127-2022).

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from oggm import cfg, utils, workflow, tasks

from oggm_adapbs import glogit, gexpit, weighted_summary, weighted_quantile
from oggm_adapbs.dynamic import (dynamic_equifinality_adapbs, dynamic_prior,
                                 run_dynamic_particle, iceboost_volume,
                                 terminus_step_km2, area_error_km2, save_netcdf)

# Plot colours (prior in grey, observations and OGGM's workflow in black)
C = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4']
GREY = '#999999'

## Glacier and data

Vernagtferner (Ötztal Alps, Austria), as in the previous notebooks, from the
OGGM preprocessed directories at level 3 (W5E5 climate, elevation band
flowlines). Level 3 contains the inversion flowlines, which the inversion
needs, and the results of OGGM's usual workflow (which we use for comparison
only).

The climate file of these directories starts in 1901: W5E5 itself covers
1979-2019, and is extended back with GSWP3-W5E5. We need it: our historical
runs start 30 years before the RGI date.

In [ ]:
cfg.initialize(logging_level='WARNING')
cfg.PATHS['working_dir'] = utils.gettempdir('adapbs_dynamic_equifinality', reset=True)

rgi_id = 'RGI60-11.00719'  # Vernagtferner
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/'
            'L3-L5_files/2025.6/elev_bands/W5E5/per_glacier/')
gdir = workflow.init_glacier_directories([rgi_id], from_prepro_level=3,
                                         prepro_border=80,
                                         prepro_base_url=base_url)[0]

# The temperature bias file of OGGM's informed calibration (centre of the temp_bias prior)
temp_bias_file = ('https://cluster.klima.uni-bremen.de/~oggm/ref_mb_params/oggm_v1.6/'
                  'w5e5_rgi6_perglacier_temp_bias_v2025.6.2.csv')

with xr.open_dataset(gdir.get_filepath('climate_historical')) as clim:
    years = clim.time.dt.year.values
print(f'RGI date: {gdir.rgi_date}, RGI area: {gdir.rgi_area_km2:.2f} km2, '
      f'climate data: {years[0]}-{years[-1]}')

# The three observations

**Geodetic MB.** The mean MB 2000-2019 (Hugonnet et al., 2021), per unit RGI
area, as in the previous notebooks. Its error includes errors which are shared
by many glaciers, and is large for a single glacier. We use 0.2 times this
error (`mb_err_factor`): a choice, which says that we trust the geodetic MB
more than the other observations.

**Area.** The RGI area, at the RGI date. To be precise, the model area at the
start of the year after the RGI date (`state_year`, the end of the RGI year),
and only where the ice is thicker than 2 m (OGGM's `area_min_h`, so that a thin
layer of ice in front of the glacier does not count).

The area is the observation we trust most, but the model cannot represent it
precisely: a flowline glacier has the right area in each elevation band, not
the right outline, and its area changes one grid point at a time. So its error
is not only the error of the outline (which is smaller), but also the mismatch
between the model and the outline (a "representation error"). It has two
terms:

$$\sigma_A^2 = (0.05 \, A_{RGI})^2 + 0.25 \, \Delta A^2$$

- 5% of the area, for the outline (its errors and its date) and for what a
  flowline glacier cannot represent. This is a choice: a smaller error would
  rule out more of the runs below, in particular more initial states;
- a discretisation term: the model area changes one grid point at a time
  ($\Delta A$, the area of one grid point at the terminus: the largest of the
  last 5 points of the main flowline, `terminus_step_km2`). The observation
  falls between two possible model areas, at most one grid point away from
  each other. With the factor 0.25, this term is half a grid point, so that
  both stay within two sigma of the observation: the model is not forced onto
  the one closest to it. It matters for small glaciers only.

**Mean ice thickness.** The IceBoost v2 volume divided by its area, against the
model volume divided by the model area (all the ice), at the same date as the
area. Why the mean thickness and not the volume?

- the volume grows with the glacier size, the mean thickness much less: a
  relative error means about the same for all glaciers;
- in the model, the volume and the area are not independent: a glacier which
  is too large has too much volume as well, and would be penalised twice.

IceBoost gives an error for each glacier (`iceboost_volume` reads it), but it
is not a measure of how accurate IceBoost is for this glacier (Maffezzoli et
al., 2026, Scientific Data, https://doi.org/10.1038/s41597-026-07744-9): it
is the spread of the IceBoost prediction when its inputs are perturbed,
summed over the glacier as if all errors were correlated (an upper bound, its
authors say), and it is smallest in steep terrain, where IceBoost is the least
reliable. It is also large: 61% of the volume for the median glacier of the
Alps. Tested against thickness measurements on glaciers left out of its
training, IceBoost does about as well as the other global models in Central
Europe (a root mean squared error of about 35 m per measurement). We use a
pragmatic choice instead: 20% of the mean thickness, for all glaciers
(`volume_rel_err=0.2`, the default; 30% would be the next value to try).
`volume_rel_err=None` uses the IceBoost error of each glacier, times
`volume_err_factor`.

IceBoost's thickness also has no clear date: it is given on the RGI outlines
(2003 here), but its inputs (DEM, velocities, thickness measurements) date
from 2010-2018. We compare it with the model at the RGI date. In the plausible
runs below, the glacier thins by about 15% between 2004 and 2012 (by 25-35% on
small glaciers): this is part of the uncertainty too.

In [ ]:
state_year = gdir.rgi_date + 1
period = gdir.settings['geodetic_mb_period']
geod = utils.get_geodetic_mb_dataframe().loc[[rgi_id]].set_index('period').loc[period]
ref_mb, ref_mb_err = geod['dmdtda'] * 1000, geod['err_dmdtda'] * 1000  # kg m-2 yr-1

area_step = terminus_step_km2(gdir.read_store('model_flowlines'))
area_sigma = area_error_km2(gdir.rgi_area_km2, area_step)

vol, vol_err, vol_area = iceboost_volume(gdir, return_area=True)
thick = vol / vol_area * 1e3  # km3 / km2 -> m

targets = pd.DataFrame({
    'observation': [ref_mb, gdir.rgi_area_km2, thick],
    'error (data)': [ref_mb_err, np.nan, vol_err / vol * thick],
    'sigma (AdaPBS)': [0.2 * ref_mb_err, area_sigma, 0.2 * thick],
}, index=['MB 2000-2019 (kg m-2 yr-1)', f'area {state_year} (km2)',
          f'mean thickness {state_year} (m)'])
print(f'Terminus grid point: {area_step:.3f} km2. '
      f'IceBoost volume: {vol:.3f} ± {vol_err:.3f} km3 ({vol_err / vol:.0%})')
targets.round(3)

- The geodetic MB: -913 ± 154 kg m-2 yr-1, used with an error of 31.
- The area: 8.56 km². The terminus grid point is 0.171 km² (2% of the area),
  so that the discretisation term barely adds anything to the 5%: sigma
  0.436 km² instead of 0.428.
- The mean thickness: 51.6 m (0.441 km³ over 8.56 km²). The IceBoost error is
  66% (34 m): anything between 17 and 86 m would be within one sigma. With
  20%, the error is 10.3 m.

# One run of the model

`run_dynamic_particle` runs the model for one parameter set (a "particle" in
the vocabulary of AdaPBS). It does the three steps of OGGM's usual workflow,
but with the parameters of the particle:

1. **Inversion.** The thickness of the glacier at the RGI date, with OGGM's
   inversion tasks (`apparent_mb_from_any_mb`, `prepare_for_inversion`,
   `mass_conservation_inversion`, `filter_inversion_output`,
   `init_present_time_glacier`). The inversion assumes a glacier in
   equilibrium: the MB parameters set the shape of the MB profile (its offset
   is removed), and therefore the ice flux, and A the ice creep. The softer the
   ice (the larger A), the thinner the ice which carries the same flux: the
   thickness scales with $A^{-1/5}$. The parameter is the log10 of a factor on
   OGGM's default A (`log10_glen_a_factor`: 0 is the default A, 1 is ten times
   A). We do not use the A of the regional calibration: this would bring the
   IceBoost volume of the region into the prior.
2. **Initial state.** The inverted glacier is the glacier *at the RGI date*. A
   simulation which starts 30 years earlier needs a glacier for that date. As
   in OGGM's dynamic spinup, we make one: the inverted glacier runs for 20
   years in a constant climate (the mean climate of the 30 years before the RGI
   date) with the particle's temperature bias plus an offset
   (`initial_state_temp_offset`). A negative offset (colder) gives a larger
   glacier, a positive one a smaller glacier. OGGM's dynamic spinup searches
   the temperature bias which gives the RGI area at the RGI date; here, the
   offset is a parameter like the others, and the observations tell which
   values are plausible.

   This is **not a reconstruction** of the glacier 30 years ago. The present
   glacier and the past climate do not determine the past glacier: many
   different past glaciers lead to the same present one (Eis et al., 2019,
   https://doi.org/10.5194/tc-13-3317-2019; Eis et al., 2021,
   https://doi.org/10.3389/feart.2021.595755). The initial state period only
   produces a range of possible starting glaciers: its offset stands for our
   ignorance of the past glacier. Its 20 years have no calendar date.
3. **Historical run** from 30 years before the RGI date (here 1974) to the end
   of the geodetic period (2020), with the historical climate and the
   particle's temperature bias (without the offset).

The particle writes its settings and its inversion to the glacier directory
(with its own file suffix, so that several particles can run at the same time)
and deletes them at the end. It returns the three predictions, the inverted
volume, the volume at the start of the historical run and the time series of
the volume, area and mass, including the initial state period.

Let us run a few particles by hand. We start from the MB parameters of OGGM's
static calibration (stored in the preprocessed directory) and the default A,
then change the initial state offset and A.

In [ ]:
mb_params = ['melt_f', 'prcp_fac', 'temp_bias']
calib = {p: gdir.settings[p] for p in mb_params}  # OGGM's static calibration
print('OGGM static calibration:', {p: round(v, 2) for p, v in calib.items()})

run_kwargs = dict(state_year=state_year, spinup_years=30, initial_state_years=20,
                  ref_period=(2000, 2020), glen_a_ref=cfg.PARAMS['inversion_glen_a'])
demo = {'OGGM calibration, default A': {},
        'initial state offset -1 K': {'initial_state_temp_offset': -1},
        'initial state offset +1 K': {'initial_state_temp_offset': 1},
        'A x 5': {'log10_glen_a_factor': np.log10(5)}}
base = {**calib, 'log10_glen_a_factor': 0., 'initial_state_temp_offset': 0.}
runs = {name: run_dynamic_particle(gdir, {**base, **change}, '_demo', **run_kwargs)
        for name, change in demo.items()}

pd.DataFrame({name: {'inverted volume (km3)': r['volume_inversion_km3'],
                     f'volume {state_year - 30} (km3)': r['volume_initial_state_km3'],
                     'MB 2000-2019 (kg m-2 yr-1)': r['pred'][0],
                     f'area {state_year} (km2)': r['pred'][1],
                     f'mean thickness {state_year} (m)': r['pred'][3],
                     'run time (s)': r['run_time']}
              for name, r in runs.items()}).T.round(3)

A plot of the volume, area and MB of a run against the observations (we reuse
it below). The volume panel shows the IceBoost volume with the error used by
AdaPBS (20%, in black) and the IceBoost error (grey); AdaPBS uses the mean
thickness, which is the volume divided by the area. The MB panel shows the
annual specific MB (the mass change of each year, per unit RGI area, as the
geodetic MB) and the geodetic MB 2000-2019 with its error. The grey area is the
initial state period.

In [ ]:
def specific_mb(mass):
    # Annual specific MB (kg m-2 yr-1, per unit RGI area) from the mass series
    return np.diff(mass, axis=-1) / gdir.rgi_area_m2


def plot_obs(axes, initial_state_years=20):
    ys = state_year - 30
    for ax in axes:
        if initial_state_years:
            ax.axvspan(ys - initial_state_years, ys, color='0.92')
        ax.set_xlabel('Year')
    axes[0].errorbar(state_year, vol, yerr=vol_err, fmt='none', color=GREY, capsize=3,
                     label='IceBoost (its error)')
    axes[0].errorbar(state_year, vol, yerr=0.2 * vol, fmt='o', color='k', capsize=3,
                     label='IceBoost (20%)')
    axes[1].errorbar(state_year, gdir.rgi_area_km2, yerr=area_sigma, fmt='o',
                     color='k', capsize=3, label='RGI')
    s = 0.2 * ref_mb_err
    axes[2].fill_between([2000, 2020], ref_mb - s, ref_mb + s, color='k', alpha=0.2)
    axes[2].plot([2000, 2020], [ref_mb, ref_mb], color='k', label='Geodetic MB')
    axes[0].set_ylabel('Volume (km$^3$)')
    axes[1].set_ylabel('Area, ice thicker than 2 m (km$^2$)')
    axes[2].set_ylabel('Specific MB (kg m$^{-2}$ yr$^{-1}$)')


def plot_run(axes, t, volume, area, mass, **kwargs):
    # One run: lines
    axes[0].plot(t, volume, **kwargs)
    axes[1].plot(t, area, **kwargs)
    axes[2].plot(t[:-1], specific_mb(mass), **kwargs)


def plot_band(axes, ds, color=C[0], label='AdaPBS', fill=True):
    # The particles of an AdaPBS run: weighted median and 5-95% range
    t = ds.time.values
    w = ds.weight.values
    series = [(t, ds.volume_km3.values), (t, ds.area_km2.values),
              (t[:-1], specific_mb(ds.mass_kg.values))]
    for ax, (tt, x) in zip(axes, series):
        ok = np.all(np.isfinite(x), axis=1)
        q = np.array([weighted_quantile(x[ok, i], w[ok], [0.05, 0.5, 0.95])
                      for i in range(x.shape[1])])
        if fill:
            ax.fill_between(tt, q[:, 0], q[:, 2], color=color, alpha=0.25, lw=0)
        else:
            ax.plot(tt, q[:, [0, 2]], color=color, lw=1, ls=':')
        ax.plot(tt, q[:, 1], color=color, label=f'{label} (median, 5-95%)')


t_demo = np.arange(state_year - 30 - 20, 2021)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_obs(axes)
for (name, r), c in zip(runs.items(), C):
    plot_run(axes, t_demo, r['volume_km3'], r['area_km2'], r['mass_kg'], color=c, label=name)
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=8)
plt.tight_layout()

- With OGGM's static calibration and the default A (blue), the inverted volume
  is 0.587 km³ (61 m mean thickness in 2004, more than IceBoost: the regional A
  of the preprocessed directory is 2.4 times larger, which gives thinner ice).
  The glacier shrinks a little during the initial state period. Its MB
  2000-2019 is -971 kg m-2 yr-1, not -913: the static calibration does not hold
  for the dynamical run. This is the mismatch which OGGM's dynamic calibration
  corrects.
- The initial state offset sets the starting glacier: 0.702 km³ in 1974 with an
  offset of -1 K, 0.389 km³ with +1 K. The differences persist until 2020. With
  +1 K, the glacier tongue thins until it disappears after 1990 (the area drops
  to 6.4 km² in 2020), and the MB 2000-2019 is less negative (-728 against
  -1028 with -1 K): a smaller glacier has less ice at low elevations.
- Five times the default A gives 27% less ice ($5^{-1/5} = 0.72$): a mean
  thickness of 43 m instead of 61 m, with the same area and MB.
- The MB during the initial state period is constant (constant climate). The
  MB of the historical run follows the climate: it is strongly positive at the
  end of the 1970s (about +1000 kg m-2 yr-1 in 1978-1979). Whether the model
  gets these years right is not tested by any of our observations, which are
  all after 2000; but the 30-year spinup goes through them.

Each run takes a few tenths of a second.

# The prior

We do not want to favour any value: the prior is wide. For each parameter, a
logit-normal centred in the middle of its bounds, with a standard deviation of
1.4 in normal space: close to uniform between the bounds (see the second
notebook).

- `melt_f`, `prcp_fac`: these are factors, like A: half or twice a value are
  equally far from it. So they vary in log10 space (`log10_melt_f`,
  `log10_prcp_fac`), and the prior is close to uniform in log10. `melt_f`:
  OGGM's bounds (1.5 to 17, prior median about 5). `prcp_fac`: 0.5 to 5
  (prior median about 1.6), narrower than OGGM's bounds (0.1 to 10): a
  precipitation factor of 0.1 or 10 on W5E5 is hard to believe, and in a first
  version, plausible runs with `prcp_fac` down to 0.17 set the spread of the
  runoff (next notebook);
- `temp_bias`: ± 10 K around the value of the temperature bias file (OGGM's
  bounds are ± 15 K, but most of these glaciers would vanish or fill the whole
  domain);
- `log10_glen_a_factor`: ± 1, from 0.1 to 10 times OGGM's default A;
- `initial_state_temp_offset`: -3 to +2 K. This one is a statement about
  which initial states we consider possible. In a first version (± 2 K), the
  plausible runs of all four glaciers we tried leaned cold, and the bound at
  -2 K cut them off: we widened the cold side. A bound which cuts the
  posterior is our choice, not something the observations say.

These bounds are choices too: a parameter which piles up against a bound in the
results below would tell us that the bound is too narrow.

`dynamic_prior` returns the default prior of `dynamic_equifinality_adapbs`; we
change its bounds and centres.

In [ ]:
tb_centre = dynamic_prior(gdir, temp_bias_file_path=temp_bias_file)['centre']['temp_bias']
s = gdir.settings
prior_bounds = {'log10_melt_f': tuple(np.log10([s['melt_f_min'], s['melt_f_max']])),
                'log10_prcp_fac': tuple(np.log10([0.5, 5])),
                'temp_bias': (tb_centre - 10, tb_centre + 10),
                'log10_glen_a_factor': (-1, 1),
                'initial_state_temp_offset': (-3, 2)}
prior_centre = {p: (lo + hi) / 2 for p, (lo, hi) in prior_bounds.items()}
prior_sd = 1.4
prior = dynamic_prior(gdir, prior_centre=prior_centre, prior_bounds=prior_bounds)
params = prior['params']

# Samples of the prior, for the plots and the table
rng = np.random.default_rng(0)
z = (glogit(prior['x_centre'], prior['x_min'], prior['x_max'])
     + prior_sd * rng.standard_normal((20000, len(params))))
prior_samples = gexpit(z, prior['x_min'], prior['x_max'])


def summary_table(theta, w, params):
    # Weighted quantiles of the parameters, and of melt_f and prcp_fac themselves
    tab = weighted_summary(theta, w, params)
    lin = {p.replace('log10_', ''): 10 ** theta[:, k] for k, p in enumerate(params)
           if p in ('log10_melt_f', 'log10_prcp_fac')}
    if lin:
        tab = pd.concat([tab, weighted_summary(np.stack(list(lin.values()), axis=1), w,
                                               list(lin))])
    return tab


prior_q = summary_table(prior_samples, np.ones(len(z)) / len(z), params)
bounds = pd.DataFrame({'min': prior['x_min'], 'max': prior['x_max']}, index=params)
bounds.loc['melt_f'] = [s['melt_f_min'], s['melt_f_max']]
bounds.loc['prcp_fac'] = [0.5, 5]
pd.concat([bounds, prior_q], axis=1).round(2)

# Running AdaPBS

`dynamic_equifinality_adapbs` reads the observations, builds the prior and
runs AdaPBS with `run_dynamic_particle` as forward model. Its main arguments:

- `spinup_years=30`, `initial_state_years=20`: the length of the historical run
  before the RGI date and of the initial state period;
- `mb_err_factor=0.2`, `area_err=0.05`, `area_k=0.25`,
  `volume_target='thickness'`, `volume_rel_err=0.2`: the observations and their
  errors described above (these are the defaults);
- `reject_sigma=5`: a particle whose *inverted* glacier is more than 5 errors
  away from the IceBoost thickness is not run further (it gets a zero weight).
  This saves the dynamical runs of hopeless particles. The thickness at the RGI
  date is not the inverted thickness (the glacier changes during the initial
  state period and the historical run), so the threshold must be generous;
- `n_particles=500`, `max_iterations=20`: with wide priors, AdaPBS needs more
  iterations than in the previous notebooks.

**Running the particles in parallel.** The runs of one iteration are
independent, so they can run at the same time:

- `n_workers`: the number of processes which run the particles of *one*
  glacier on one computer (here 6; leave a core or two free). In a script
  (not in a notebook), put the code under `if __name__ == '__main__':`: each
  process starts a new Python, which imports the script.
- For many glaciers: one job per glacier on a cluster, each with its own
  `n_workers` (see the scripts in `scripts/`), or OGGM's multiprocessing over
  the glaciers with `n_workers=1`. Not both at the same time: OGGM's
  processes cannot start processes of their own.
- Never run two AdaPBS runs on the same glacier directory at the same time:
  their particles would write files with the same names.

**Saved runs.** An AdaPBS run takes several minutes, so we save it to
`out/wide_priors/<rgi_id>_adapbs.nc` (the folder `out/` is not tracked by
git), and the next time the notebook loads it instead of running AdaPBS again.
**Delete the file after changing the settings**, otherwise the old run is
loaded. The script `scripts/run_glacier.py` writes the same files with the
same default settings: a run made with the script (on a cluster, for example)
can be analysed here.

In [ ]:
adapbs_kwargs = dict(prior_centre=prior_centre, prior_bounds=prior_bounds,
                     prior_sd=prior_sd, temp_bias_file_path=temp_bias_file,
                     mb_err_factor=0.2, volume_rel_err=0.2,
                     n_particles=500, max_iterations=20, seed=0, n_workers=6)

def run_or_load(fpath, **kwargs):
    # Load the AdaPBS run from fpath if it exists, else run it and save it.
    # The settings are saved with the run: a run made with other settings is
    # not loaded silently.
    settings = repr({k: v for k, v in sorted(kwargs.items()) if k != 'n_workers'})
    if os.path.exists(fpath):
        with xr.open_dataset(fpath) as d:
            out = d.load()
        if out.attrs.get('notebook_settings', settings) != settings:
            raise ValueError(f'{fpath} was made with other settings: '
                             'delete it to run AdaPBS again.')
        print(f'Loaded from {fpath} (delete the file to run AdaPBS again)')
    else:
        t0 = time.time()
        out = dynamic_equifinality_adapbs(gdir, **kwargs)
        out.attrs['wall_time_s'] = time.time() - t0
        out.attrs['notebook_settings'] = settings
        os.makedirs(os.path.dirname(fpath), exist_ok=True)
        save_netcdf(out, fpath)
    print(f'{out.sizes["particle"]} runs, {len(out.adapbs_iteration)} iterations, '
          f'run time {out.attrs["wall_time_s"]:.0f} s')
    return out


ds = run_or_load(f'out/wide_priors/{rgi_id}_adapbs.nc', spinup_years=30,
                 initial_state_years=20, **adapbs_kwargs)
ds

The Dataset has the same structure as in the previous notebooks, with the
predictions of the three observations in `pred` (dimension `obs`), and more
per-particle output: the Glen A, the inverted volume, the volume at the start of
the historical run, the volume, area and mass time series (dimension `time`,
starting with the initial state period), the `status` of the run (`ok`,
`rejected`, or the error) and its run time.

The diagnostics per iteration, and the status of the runs:

In [ ]:
def diagnostics(ds):
    d = ds[[v for v in ds.data_vars if v.startswith('adapbs_')]].to_dataframe()
    d.columns = [c.replace('adapbs_', '') for c in d.columns]
    return d

def status(ds):
    s = pd.Series(ds.status.values)
    s[~s.isin(['ok', 'rejected', 'disappeared'])] = 'error'
    return s.value_counts()

display(diagnostics(ds).round(3))
status(ds)

AdaPBS needed 7 iterations (3500 runs). This run was made on our cluster with
`scripts/run_glacier.py` (same settings, 96 s on 32 processes); in the notebook
it takes a few minutes on 6 processes. The effective sample size (`neff`, the
number of runs which really count given their weights) grows with the
iterations, and AdaPBS stops when it reaches 30% of the 500 particles (here 33%
after the 7th).

Half of the particles of the first iteration (261 of 500) gave no prediction:

- 146 glaciers disappeared before the RGI date (`disappeared`), mostly with a
  warm `temp_bias`: there is nothing to compare with the observed area. (A
  glacier which disappears *after* the RGI date is a valid run: it could be
  the truth);
- 80 outgrew the domain (`Glacier exceeds domain boundaries`, an `error`);
- 35 inverted glaciers were more than 5 errors from the IceBoost thickness
  (`rejected`): they are not run further.

They all get a zero weight. After a few iterations, AdaPBS no longer proposes
such parameter sets.

# The plausible parameter sets

The weighted particles are the plausible parameter sets. How do they compare
with the prior?

In [ ]:
def particles(ds):
    # All the particles (n, n_params) and their weights
    p = ds.attrs['calibrate_params']
    return np.stack([ds[k].values for k in p], axis=1), ds.weight.values

theta, w = particles(ds)
summary = pd.concat({'prior': prior_q, 'posterior': summary_table(theta, w, params)},
                    axis=1)
summary.round(2)

In [ ]:
labels = {'melt_f': 'melt_f (kg m$^{-2}$ day$^{-1}$ K$^{-1}$)', 'prcp_fac': 'prcp_fac',
          'log10_melt_f': 'log10 of melt_f', 'log10_prcp_fac': 'log10 of prcp_fac',
          'temp_bias': 'temp_bias (K)', 'log10_glen_a_factor': 'log10 of the A factor',
          'initial_state_temp_offset': 'initial_state_temp_offset (K)'}
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for k, (p, ax) in enumerate(zip(params, axes)):
    bins = np.linspace(prior['x_min'][k], prior['x_max'][k], 31)
    ax.hist(prior_samples[:, k], bins=bins, density=True, color=GREY, alpha=0.5,
            label='prior')
    ax.hist(theta[:, k], bins=bins, weights=w, density=True, histtype='step',
            color=C[0], lw=2, label='posterior')
    ax.set_xlabel(labels[p])
    ax.set_yticks([])
axes[0].legend(fontsize=9)
plt.tight_layout()

- `melt_f` and `prcp_fac` (in the table, also in their own units): low values
  of `melt_f` are favoured (median 3.4; prior median 5.1), but the range stays
  wide: from 1.8 to 11.4 (5-95%). `prcp_fac` is barely constrained (median
  1.23, from 0.62 to 3.5; prior median 1.59, from 0.62 to 4.1). The
  equifinality of the MB parameters (second notebook) is still there.
- `temp_bias`: from -1.5 to 5.4 K (median 1.9), much narrower than the prior
  (± 10 K): very cold or very warm biases give glaciers which outgrow the
  domain or disappear.
- Glen's A: the posterior is almost the prior (5-95%: -0.79 to 0.66, prior
  -0.82 to 0.81). The mean thickness, even with an error of 20%, does not tell
  which A is right here. We see why below.
- The initial state offset: offsets warmer than about +1.1 K are ruled out
  (the glacier shrinks too much), colder offsets are not: the cold side
  follows the prior down to -3 K.

The histograms are noisy: about 165 runs count (`neff`). More particles or
several seeds would smooth them (see the open questions at the end).

In [ ]:
# Equally weighted particles, resampled with the posterior weights
idx = np.random.default_rng(1).choice(len(w), 2000, p=w)
post = ds.isel(particle=idx)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
panels = [(post['melt_f'], post['prcp_fac'], post['temp_bias'],
           labels['melt_f'], labels['prcp_fac'], labels['temp_bias']),
          (post['log10_glen_a_factor'], post['volume_inversion_km3'], post['melt_f'],
           labels['log10_glen_a_factor'], 'inverted volume (km$^3$)', labels['melt_f']),
          (post['log10_glen_a_factor'], post['initial_state_temp_offset'], post['pred'][:, 2],
           labels['log10_glen_a_factor'], labels['initial_state_temp_offset'],
           f'mean thickness {state_year} (m)')]
for ax, (x, y, c, xl, yl, cl) in zip(axes, panels):
    sc = ax.scatter(x, y, c=c, s=4, cmap='viridis')
    plt.colorbar(sc, ax=ax, label=cl)
    ax.set_xlabel(xl)
    ax.set_ylabel(yl)
plt.tight_layout()

- Left: the equifinality of the MB parameters, as in the second notebook: a
  thin surface in the space of the three parameters, along which all
  combinations give the same MB 2000-2019.
- Middle: A sets the inverted volume (5-95%: 0.40 to 0.75 km³). The MB
  parameters feed into the inversion as well: for the same A, a higher
  `melt_f` (yellow) means a larger mass turnover, a larger ice flux, and
  thicker ice. But A dominates: thickness scales with the flux to the power
  1/5.
- Right: why A stays unconstrained. With stiff ice (low A), the inverted
  glacier is thick, and a warm initial state makes it thinner before 2004.
  With soft ice, the inverted glacier is thin, and the initial state can be
  colder. A and the initial state offset compensate each other (rank
  correlation -0.39 among the plausible runs): the observations constrain a
  combination of the two, not each of them.

# The fit to the observations

The predictions of the particles, before (the first iteration, which samples the
prior) and after AdaPBS (weighted), with the observations and the errors
used by AdaPBS:

In [ ]:
obs_labels = ['MB 2000-2019 (kg m$^{-2}$ yr$^{-1}$)', f'Area {state_year} (km$^2$)',
              f'Mean thickness {state_year} (m)']

def pred_summary(ds):
    ok = np.isfinite(ds.pred[:, 0]).values
    q = [weighted_quantile(ds.pred[ok, k].values, ds.weight.values[ok], [0.05, 0.5, 0.95])
         for k in range(3)]
    return pd.DataFrame(q, index=['MB', 'area', 'thickness'], columns=['q5', 'q50', 'q95'])

first = (ds.iteration == 1).values
ok = np.isfinite(ds.pred[:, 0]).values
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for k, ax in enumerate(axes):
    lo, hi = np.quantile(ds.pred[first & ok, k], [0.02, 0.98])
    bins = np.linspace(lo, hi, 50)
    ax.hist(ds.pred[first & ok, k], bins=bins, density=True, color=GREY, alpha=0.5,
            label='prior (iteration 1)')
    ax.hist(ds.pred[ok, k], bins=bins, weights=w[ok], density=True, histtype='step',
            color=C[0], lw=2, label='posterior')
    o, s = float(ds.ref_obs[k]), float(ds.ref_obs_err[k])
    ax.axvline(o, color='k', label='observation')
    ax.axvspan(o - s, o + s, color='k', alpha=0.1)
    ax.set_xlabel(obs_labels[k])
    ax.set_yticks([])
axes[0].legend(fontsize=9)
plt.tight_layout()

pd.concat({'observation': pd.DataFrame({'value': ds.ref_obs.values,
                                        'sigma': ds.ref_obs_err.values},
                                       index=['MB', 'area', 'thickness']),
           'posterior': pred_summary(ds)}, axis=1).round(3)

The plausible runs match the three observations: the MB (-914 kg m-2 yr-1,
5-95% -964 to -861), the area (8.39 km², 8.10 to 9.07, slightly below the RGI
area) and the mean thickness (51.5 m, 35 to 65). Before AdaPBS (the first
iteration), the area ranged from 0 (glaciers which disappeared) to about
12 km², and the mean thickness from a few metres to about 150 m.

# The plausible runs, and OGGM's usual workflow

How do the plausible glaciers evolve? For comparison, we run OGGM's usual
workflow on the same glacier: the static calibration and the inversion (with
the regional A) of the preprocessed directory, then OGGM's dynamic calibration
(`run_dynamic_melt_f_calibration`), with the same spinup start (1974), the same
area definition (ice thicker than 2 m) and the same MB error (0.2 times the
Hugonnet error). It changes the glacier directory (settings, inversion), so we
run it on a copy.

In [ ]:
ref = tasks.copy_to_basedir(gdir, base_dir=utils.gettempdir('adapbs_dynamic_ref', reset=True),
                            setup='all')
ref.settings['min_ice_thick_for_area'] = 2.
tasks.run_dynamic_melt_f_calibration(ref, ys=state_year - 30, ye=2020,
                                     ref_mb_err_scaling_factor=0.2,
                                     output_filesuffix='_dyncal',
                                     kwargs_run_function={'min_ice_thickness': 2.},
                                     kwargs_fallback_function={'min_ice_thickness': 2.})
with xr.open_dataset(ref.get_filepath('model_diagnostics', filesuffix='_dyncal')) as d:
    oggm_run = d.load()
oggm_run['time'] = np.round(oggm_run.time).astype(int)

diag = ref.get_diagnostics()
oggm = {'melt_f': ref.settings['melt_f'], 'prcp_fac': ref.settings['prcp_fac'],
        'temp_bias': ref.settings['temp_bias'],
        'log10_glen_a_factor': np.log10(ref.settings['inversion_glen_a'] /
                                        cfg.PARAMS['inversion_glen_a']),
        'spinup temp_bias': diag['temp_bias_dynamic_spinup']}
mass = oggm_run.mass_kg
oggm_pred = {'MB': float(mass.sel(time=2020) - mass.sel(time=2000)) / gdir.rgi_area_m2 / 20,
             'area': float(oggm_run.area_min_h_m2.sel(time=state_year)) * 1e-6,
             'thickness': float(oggm_run.volume_m3.sel(time=state_year) /
                                oggm_run.area_m2.sel(time=state_year))}
print(diag['used_spinup_option'])
display(pd.Series(oggm).round(3).to_frame('OGGM'))
pd.concat([pd.Series(oggm_pred).to_frame('OGGM'), pred_summary(ds)], axis=1).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_obs(axes)
plot_band(axes, ds)
plot_run(axes, oggm_run.time.values, oggm_run.volume_m3.values * 1e-9,
         oggm_run.area_min_h_m2.values * 1e-6, oggm_run.mass_kg.values,
         color='k', ls='--', label='OGGM dynamic calibration')
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=8)
plt.tight_layout()

OGGM's dynamic calibration changed `melt_f` from 5.33 (static) to 5.16, and
found a spinup temperature bias of 1.64 K (0.07 K below `temp_bias`, for a
spinup period of 30 years where ours is 20). Its glacier matches the three
observations (MB -905 kg m-2 yr-1, area 8.55 km², mean thickness 53.7 m), and
lies within the ranges of the plausible runs: OGGM's usual workflow picks one
of them.

The plausible runs add the spread:

- volume: in 1974, the plausible glaciers start between 0.39 and 0.67 km³
  (5-95%); in 2004, between 0.29 and 0.57 km³; in 2020, between 0.14 and
  0.42 km³: a factor of 3, for glaciers which all match the observations;
- area: close to the RGI area until 2004, then the range widens: by 2020, the
  5% quantile is down to 7.3 km². These are thin glaciers, whose tongue melts
  away;
- MB: all runs follow the same climate, but the amplitude of the year-to-year
  changes varies, probably because runs with a high `melt_f` are more
  sensitive to temperature. Their mean over 2000-2019 is the same.

These differences are invisible in the observations, but they matter for what
the glacier does next: the next notebook looks at its runoff.

# How much does the initial state matter?

The initial state is the least known part of the workflow. What if we leave it
out? With `initial_state_years=0`, the historical run starts in 1974 from the
glacier inverted at the RGI date (the glacier of 2004 is put in 1974), and
`initial_state_temp_offset` does not vary (four parameters). This is not a
better assumption about the past glacier, just a different one: it shows how
much our conclusions depend on it.

In [ ]:
ds_no_init = run_or_load(f'out/wide_priors_no_initial_state/{rgi_id}_adapbs.nc',
                         spinup_years=30, initial_state_years=0,
                         calibrate_params=params[:4], **adapbs_kwargs)
theta_ni, w_ni = particles(ds_no_init)
pd.concat({'prior': prior_q,
           'with initial state period': summary['posterior'],
           'without': summary_table(theta_ni, w_ni, params[:4])},
          axis=1).round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for k, (p, ax) in enumerate(zip(params[:4], axes)):
    bins = np.linspace(prior['x_min'][k], prior['x_max'][k], 31)
    ax.hist(prior_samples[:, k], bins=bins, density=True, color=GREY, alpha=0.5,
            label='prior')
    for d, c, lab in [(ds, C[0], 'with initial state period'),
                      (ds_no_init, C[1], 'without')]:
        ax.hist(d[p].values, bins=bins, weights=d.weight.values, density=True,
                histtype='step', color=c, lw=2, label=lab)
    ax.set_xlabel(labels[p])
    ax.set_yticks([])
axes[0].legend(fontsize=9)
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
plot_obs(axes)
plot_band(axes, ds, label='with initial state period')
plot_band(axes, ds_no_init, color=C[1], label='without', fill=False)
axes[0].legend(fontsize=8)
axes[2].legend(fontsize=8)
plt.tight_layout()

pd.DataFrame({'with initial state period': [len(ds.adapbs_iteration),
                                            ds.attrs['log_evidence']],
              'without': [len(ds_no_init.adapbs_iteration),
                          ds_no_init.attrs['log_evidence']]},
             index=['iterations', 'log evidence']).round(2)

Without the initial state period:

- AdaPBS needs 8 iterations (4000 runs, 87 s on 32 processes of our cluster);
- the MB parameters barely change;
- Glen's A is a little better constrained (5-95%: -0.57 to 0.78, median 0.14,
  i.e. 1.4 times the default A): the stiffest ice is ruled out. Without the
  offset, nothing compensates A: the thickness in 2004 depends directly on the
  inverted thickness. (In an earlier version of this notebook, with linear
  priors on `melt_f` and `prcp_fac`, A was clearly constrained without the
  initial state period and not with it; with a larger thickness error, it was
  constrained in neither: what we learn about A depends on these choices.);
- the area stays closer to the RGI area (median 8.41 km² in 2004): the
  historical run starts from the RGI outline, and the model area of this
  glacier changes little in 30 years.

The log evidence (how well a model explains the observations, all parameter
values considered) is a little higher without the initial state period (-13.62
against -14.06, a factor $e^{0.44} \approx 1.6$). This is weak, and the
estimates are noisy with about 165 runs which count: the observations do not
need the extra freedom of the initial state, but they do not rule it out
either. With one area at one date, nothing tells how different the glacier was
in 1974.

# Summary

- One AdaPBS run gives the plausible parameter sets of OGGM's whole workflow
  (MB calibration, inversion, initial state of the spinup) at once. On a
  glacier of 8.6 km², with wide priors, this takes 3500 dynamical runs with the
  initial state period (96 s on 32 processes of our cluster, a few minutes on
  6), and 4000 without.
- The plausible runs match the three observations, and OGGM's usual workflow
  gives one of them.
- The model is equifinal: the MB parameters (as with the geodetic MB alone),
  and Glen's A and the initial state offset, which compensate each other. The
  plausible runs agree on the observations, by construction, but differ
  elsewhere: the glacier before 2000 and after 2004 (a factor of 3 in the
  volume of 2020), the sensitivity of the MB to temperature.
- What the observations say about A depends on choices we cannot test: the
  thickness error and the initial state.

Open questions:

- The initial state: should a study of OGGM's equifinality include it, or fix
  it (e.g. no initial state period)? It is OGGM's usual workflow, but it opens
  a question which the observations cannot answer. On the four glaciers we
  tried, the plausible runs lean towards cold offsets (a larger glacier in
  1974 than the inverted one), as OGGM's dynamic spinup assumes.
- The spinup runs through 1974-2000, when the MB of the model is strongly
  positive at times, and none of the observations is before 2000. Observations
  of the past (older outlines, geodetic MB before 2000) would test it. The
  model also remembers its initial state for longer than 30 years.
- The choices: 0.2 times the geodetic MB error, 20% for the mean thickness
  (or 30%?), 5% for the area, the bounds of the prior (`prcp_fac` 0.5 to 5,
  the offset -3 to +2 K), the length of the spinup (30 years) and of the
  initial state period (20 years).
- Other glaciers: larger ones (Aletsch: much longer runs), and many glaciers on
  a cluster.
- Convergence: with about 165 runs which count, the histograms are noisy. More
  particles, several seeds, or a higher `neff_threshold`.

The next notebook (`dynamic_runoff_sensitivity.ipynb`) starts from the runs
saved here.